In [1]:
# ==========================================
# PASO 1: IMPORTACIONES Y LISTAS VACÍAS
# ==========================================
import spacy        # Detecta nombres y entidades en el texto
import re           # Busca patrones como emails y teléfonos
import os           # Lee carpetas y arma rutas de archivos
import pandas as pd # Construye la tabla final y exporta el CSV
import pdf2txt      # Script auxiliar que convierte PDF a texto

nlp = spacy.load("es_core_news_sm")  # Carga el modelo de español

# Una lista por dato — cada posición [0],[1]... es un CV distinto
names  = []
phones = []
emails = []
skills = []

print("✅ Celda 1 lista: Herramientas cargadas y listas preparadas.")

✅ Celda 1 lista: Herramientas cargadas y listas preparadas.


In [2]:
import pdf2txt

In [3]:
# ==========================================
# PASO 2: FUNCIÓN PARA CONVERTIR PDF A TEXTO
# ==========================================
def convert_pdf(f):
    # Construye la ruta de salida: output/txt/nombre_cv.txt
    output_filename = os.path.basename(os.path.splitext(f)[0]) + ".txt"
    output_filepath = os.path.join("output/txt", output_filename)

    # Llama a pdf2txt para extraer el texto del PDF y guardarlo en disco
    pdf2txt.main(args=[f, "--outfile", output_filepath])

    # Lee y devuelve el texto. utf-8 evita errores con tildes y eñes.
    return open(output_filepath, encoding="utf-8").read()

print("✅ Celda 2 lista: Máquina lectora de PDFs construida.")

✅ Celda 2 lista: Máquina lectora de PDFs construida.


In [4]:
# Palabras que spaCy puede etiquetar como PER pero son secciones del CV
NO_SON_NOMBRES = {
    "experiencia", "educación", "educacion", "habilidades",
    "perfil", "contacto", "resumen", "idiomas", "proyectos"
}

def es_nombre_valido(texto):
    palabras = texto.lower().split()
    if len(palabras) < 2:                          # Un nombre tiene al menos 2 palabras
        return False
    if any(p in NO_SON_NOMBRES for p in palabras): # Descarta secciones del CV
        return False
    if any(c.isdigit() for c in texto):            # Un nombre no tiene números
        return False
    return True

def parse_content(txt):
    # Patrones de búsqueda
    skillset  = re.compile(r"python|java|sql|hadoop|tableau|pandas|numpy|spacy|aws|docker|kubernetes|jenkins|node\.js|react|html|css")
    phone_num = re.compile(r"\+?\d[\d\s\-]{8,15}\d")
    email_pat = re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}")

    # spaCy analiza el texto y detecta entidades (nombres, lugares, etc.)
    doc = nlp(txt)

    # 1. NOMBRE — busca entidades PER solo en los primeros 400 caracteres (encabezado del CV)
    nombres_encontrados = [
        entity.text for entity in doc.ents
        if entity.label_ == "PER"
        and entity.start_char < 400       # Descarta entidades fuera del encabezado
        and es_nombre_valido(entity.text) # Descarta falsos positivos
    ]
    name = nombres_encontrados[0] if nombres_encontrados else "No encontrado"

    # 2. EMAIL — Regex en lugar de word.like_email, que falla si spaCy tokeniza mal el email
    emails_encontrados = re.findall(email_pat, txt)
    email = emails_encontrados[0] if emails_encontrados else "No encontrado"

    # 3. TELÉFONO — Regex acepta formatos como +54 9 11 1234-5678
    telefonos_encontrados = re.findall(phone_num, txt)
    phone = str(telefonos_encontrados[0]).strip() if telefonos_encontrados else "No encontrado"

    # 4. SKILLS — busca coincidencias y elimina duplicados con set()
    skills_list = re.findall(skillset, txt.lower())
    unique_skills_list = list(set(skills_list))

    # Agrega los resultados de este CV a las listas globales
    names.append(name)
    emails.append(email)
    phones.append(phone)
    skills.append(unique_skills_list)

✅ Celda 3 lista: Máquina extractora construida.


In [ ]:
for file in os.listdir("resumes/"):

    if file.endswith(".pdf"):  # Ignora cualquier archivo que no sea PDF
        print("\nLeyendo documento: " + file)

        ruta_completa = os.path.join("resumes/", file)  # Ej: "resumes/cv_valentin.pdf"

        txt = convert_pdf(ruta_completa)  # Paso A: PDF → texto plano
        parse_content(txt)                # Paso B: texto → datos extraídos

In [ ]:
# Arma una tabla donde cada fila es un CV y cada columna un dato extraído
df = pd.DataFrame({
    "Nombre":    names,
    "Email":     emails,
    "Teléfono":  phones,
    "Skills":    skills
})

df.to_csv("output/csv/resultados.csv", index=False, encoding="utf-8")
df  # Muestra la tabla en el notebook